# Project: Predicting Unsafe NYC Building Facades

**Owner:** Whole team

- Course: CS 513 (Data Mining & Machine Learning), Fall 2026
- Group No.: <fill in>
- Team members:
  1. <fill in>
  2. <fill in>
  3. <fill in>
  4. <fill in>

This is the notebook to hand in. It tells the whole story in order. Run the four model notebooks and `05_comparison.ipynb` first, then run this one top to bottom so the outputs are saved in the file.

Rules that apply to every notebook: use `prep` and `evaluate` for the split, the preparation and the scoring. Do not write your own. The test set is only touched inside `evaluate.run_feature_steps`.

## Problem statement

NYC requires buildings taller than six stories to have their facades inspected every five years (FISP / Local Law 11). Each facade is rated Safe, SWARMP (safe with a repair and maintenance program) or Unsafe.

**Question:** can building traits and construction-safety history predict an Unsafe rating, so the city and owners know which buildings to look at first?

<Teammate 1: replace with the final wording.>

## Data sources

|Dataset|Publisher|What we use it for|Join key|
|:--|:--|:--|:--|
|DOB NOW: Safety, Facades Compliance Filings|NYC Dept. of Buildings|The rating (label) and filing traits|BIN, borough-block-lot|
|PLUTO|NYC Dept. of City Planning|Building age, size, class, landmark status|BBL|
|DOB ECB Violations|NYC Dept. of Buildings|Violation history before each inspection|BIN|

<Teammate 2: add the row counts and the date each file was downloaded.>

## Dataset description

One row is one initial facade filing (one building in one inspection cycle).

|Variable|Details|Source|
|:--|:--|:--|
|is_unsafe|1 if the filing was rated Unsafe, else 0 (the label)|Filings|
|f_cycle|Inspection cycle|Filings|
|f_borough|Borough|Filings|
|f_prior_status|The building's rating in its previous cycle|Filings|
|f_wall_*, f_mat_*|0-1 flags for the most common wall types and materials|Filings|
|p_age_at_filing|Building age in years at the filing date|PLUTO|
|p_numfloors, p_bldgarea, p_unitstotal|Floors, floor area, number of units|PLUTO|
|p_bldgclass|First letter of the building class|PLUTO|
|p_landmark, p_histdist|1 if a landmark / in a historic district|PLUTO|
|p_ever_altered, p_years_since_alter|Alterations before the filing year|PLUTO|
|v_total, v_class1, v_construction|Violations issued before the filing date: all, Class 1, construction type|ECB|
|v_last5yrs, v_years_since_last, v_any|Recent violation history before the filing date|ECB|

<Update this table if the real columns differ.>

## Data exploration

In [ ]:
# Course      : CS 513 - Data Mining & Machine Learning
# First Name  : <fill in>
# Last Name   : <fill in>
# Id          : <fill in>
# Purpose     : The single report notebook the team submits: problem, data, exploration, preparation, results and conclusion.
#
# AI NOTICE   : The starter code in this notebook was generated by AI (Claude).
#               Prompt and context: docs/AI_PROMPT_LOG.md, entry 1.
#               Mark any cell you add with AI help the same way.

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import config
from src import evaluate, plots, prep

df = prep.load_table()
train, test = prep.split_by_building(df)
print(f"{len(df):,} filings, {df['bin'].nunique():,} buildings, "
      f"{df['is_unsafe'].mean():.1%} Unsafe")
print(f"Train: {len(train):,} rows   Test: {len(test):,} rows")

**Sample data**

In [ ]:
df.head()

In [ ]:
df.describe().T.round(2)

**Missing values**

In [ ]:
missing = df.isna().mean().sort_values(ascending=False)
missing[missing > 0].map("{:.1%}".format).to_frame("share missing")

**How many filings got each rating?**

In [ ]:
counts = df["rating"].value_counts().reindex(["SAFE", "SWARMP", "UNSAFE"])
fig, ax = plots.new_axes("Initial filings by rating", xlabel="Filings", size=(7.5, 2.8))
plots.bars(ax, counts.index, counts.values)
plt.show()
(counts / counts.sum()).map("{:.1%}".format)

**Where is the Unsafe share highest?** The dashed line is the overall share.

In [ ]:
plots.rate_bars(df, "f_prior_status", "Unsafe share by the previous cycle's rating")
plt.show()
plots.rate_bars(df, "f_borough", "Unsafe share by borough")
plt.show()
plots.rate_bars(df, "f_cycle", "Unsafe share by inspection cycle")
plt.show()

In [ ]:
age_band = pd.cut(df["p_age_at_filing"], [0, 25, 50, 75, 100, 300],
                  labels=["0-25 yrs", "26-50", "51-75", "76-100", "over 100"])
plots.rate_bars(df.assign(age_band=age_band), "age_band", "Unsafe share by building age")
plt.show()
viol_band = pd.cut(df["v_total"], [-1, 0, 2, 5, 10_000], labels=["0", "1-2", "3-5", "6 or more"])
plots.rate_bars(df.assign(viol_band=viol_band), "viol_band",
                "Unsafe share by number of earlier violations")
plt.show()

**What the exploration shows**

<Teammates 1 and 3: three or four bullet points.>

## Data preparation

1. Keep initial filings only. Amended and subsequent filings reveal the result.
2. Keep one filing per building per cycle.
3. Join PLUTO on borough-block-lot and violations on BIN.
4. Count only violations issued before the filing date.
5. Split 80/20 **by building**, so no building is in both train and test.
6. Inside each training fold: fill missing values, scale numbers, one-hot encode text.

The code is in `src/build_table.py` and `src/prep.py`.

In [ ]:
overlap = set(train["bin"]) & set(test["bin"])
print(f"Buildings in both train and test: {len(overlap)}")
print(f"Unsafe share  train: {train['is_unsafe'].mean():.1%}   test: {test['is_unsafe'].mean():.1%}")

## Predictive models

Each model was trained three times: on filing features only, then adding PLUTO, then adding violation history. Tuning used 5-fold cross-validation grouped by building. The code for each model is in its own notebook:

1. Naive Bayes (`01_naive_bayes.ipynb`)
2. kNN and distance-weighted kNN (`02_knn.ipynb`)
3. CART and Random Forest (`03_decision_tree.ipynb`)
4. Neural network (`04_neural_network.ipynb`)
5. Baselines and SMOTE (`05_comparison.ipynb`)

<Each owner: paste your key chart and two sentences here.>

## Summary

In [ ]:
table = pd.read_csv(config.COMPARISON_PATH)
cols = ["model", "step", "roc_auc", "recall", "precision", "f1", "accuracy"]
print("All models, all feature steps, same test set (sorted by ROC AUC)")
table[cols].sort_values("roc_auc", ascending=False).round(3).reset_index(drop=True)

In [ ]:
table.pivot(index="model", columns="step", values="roc_auc").round(3)

In [ ]:
fig, ax = plots.comparison_chart(table)
plt.show()

## Conclusion

<Whole team: which model won, whether the added data helped, what a building owner or the city could do with this, the limits of the study, and future work.>

## References

- NYC Department of Buildings. *DOB NOW: Safety, Facades Compliance Filings.* NYC Open Data, dataset xubg-57si.
- NYC Department of City Planning. *Primary Land Use Tax Lot Output (PLUTO).* NYC Open Data, dataset 64uk-42ks.
- NYC Department of Buildings. *DOB ECB Violations.* NYC Open Data, dataset 6bgk-3dad.
- Published filings-only reference model: https://huggingface.co/iaac-maai/day2-team1-unsafe-facade
- AI use: see `docs/AI_PROMPT_LOG.md`.